# Customer Segmentation Analysis

**Oasis Infobyte Data Analytics Internship**  
**Track:** Data Analytics | **Level:** L2  
**Project:** Customer Segmentation Analysis

---

## 1. Project Introduction

This project analyzes customer data from a shopping mall to identify distinct customer segments using unsupervised machine learning techniques. By understanding customer behavior patterns, businesses can develop targeted marketing strategies, optimize product offerings, and improve customer retention.

Customer segmentation is the practice of dividing a customer base into groups of individuals that are similar in specific ways relevant to marketing, such as age, gender, income, and spending habits.

---

## 2. Business Objective

The primary objectives of this analysis are:

1. **Identify distinct customer segments** within the mall's customer base using K-Means clustering.
2. **Analyze key customer attributes** including Age, Annual Income, and Spending Score.
3. **Develop actionable marketing strategies** tailored to each identified segment.
4. **Maximize business value** by understanding which segments contribute most to revenue and which require targeted engagement.
5. **Provide data-driven recommendations** for improving customer experience and loyalty programs.

---

## 3. Dataset Description

**Dataset:** Mall_Customers.csv  
**Records:** 200 customers  
**Attributes:** 5 features

| Feature | Type | Description |
|---------|------|-------------|
| CustomerID | Integer | Unique identifier for each customer |
| Gender | Categorical | Male or Female |
| Age | Integer | Customer age (18-70 years) |
| Annual Income (k$) | Integer | Annual income in thousands of dollars |
| Spending Score (1-100) | Integer | Score assigned by the mall based on customer behavior and spending nature |

---

## 4. Import Libraries

Let's import all the necessary libraries for data manipulation, visualization, and machine learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['font.size'] = 10

## 5. Load Dataset

Loading the Mall Customers dataset from the data directory.

In [ ]:
df = pd.read_csv('data/Mall_Customers.csv')
print(f"Dataset loaded successfully!\nShape: {df.shape}")

## 6. Data Inspection

Let's examine the dataset structure, data types, and basic statistics.

In [ ]:
print("="*50)
print("DATASET OVERVIEW")
print("="*50)
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData Types:")
print(df.dtypes)
print("\n" + "="*50)
print("FIRST 10 ROWS")
print("="*50)
print(df.head(10))
print("\n" + "="*50)
print("STATISTICAL SUMMARY")
print("="*50)
print(df.describe().T)

## 7. Missing Value Analysis

Checking for missing values in the dataset.

In [ ]:
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing Values': missing_values,
    'Missing Percentage': missing_percentage
})

print("="*50)
print("MISSING VALUE ANALYSIS")
print("="*50)
print(missing_df)
print(f"\nTotal Missing Values: {missing_values.sum()}")

## 8. Duplicate Check

Checking for duplicate records in the dataset.

In [ ]:
duplicates = df.duplicated().sum()
print("="*50)
print("DUPLICATE CHECK")
print("="*50)
print(f"Total Duplicate Rows: {duplicates}")

if duplicates > 0:
    print("\nDuplicate rows found:")
    print(df[df.duplicated(keep=False)])
else:
    print("\nNo duplicate records found. Dataset is clean.")

## 9. Data Cleaning

Based on our inspection, the dataset is clean with no missing values or duplicates. Let's verify unique values in categorical columns and remove unnecessary columns like CustomerID for analysis purposes.

In [ ]:
print("="*50)
print("UNIQUE VALUES IN CATEGORICAL COLUMNS")
print("="*50)
for col in df.select_dtypes(include=['object']).columns:
    print(f"\n{col}:")
    print(df[col].value_counts())

df_analysis = df.drop('CustomerID', axis=1)
print("\nCustomerID column dropped for analysis (unique identifier, not predictive).")

## 10. Exploratory Data Analysis (EDA)

### 10.1 Gender Distribution

Understanding the gender composition of our customer base helps in tailoring marketing messages and product placements.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#FF6B6B', '#4ECDC4']
gender_counts = df['Gender'].value_counts()

wedges, texts, autotexts = ax.pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%', 
                                    colors=colors, startangle=90, explode=(0.02, 0.02),
                                    textprops={'fontsize': 12, 'fontweight': 'bold'})

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(13)

ax.set_title('Gender Distribution of Mall Customers', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('outputs/gender_distribution.png', bbox_inches='tight')
plt.close()

print(f"Female Customers: {gender_counts['Female']} ({gender_counts['Female']/len(df)*100:.1f}%)")
print(f"Male Customers: {gender_counts['Male']} ({gender_counts['Male']/len(df)*100:.1f}%)")
print("\nInsight: Female customers form the majority, suggesting a stronger female-oriented product mix may be beneficial.")

### 10.2 Age Distribution

Age distribution reveals the target demographic for the mall and helps in understanding which age groups frequent the mall most.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.histplot(df['Age'], bins=20, kde=True, color='#3498db', ax=axes[0], edgecolor='white', linewidth=0.5)
axes[0].axvline(df['Age'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["Age"].mean():.1f}')
axes[0].axvline(df['Age'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df["Age"].median():.1f}')
axes[0].set_title('Age Distribution (Histogram)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Frequency')
axes[0].legend()

sns.boxplot(x='Gender', y='Age', data=df, palette=['#FF6B6B', '#4ECDC4'], ax=axes[1])
axes[1].set_title('Age Distribution by Gender', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Age')

plt.tight_layout()
plt.savefig('outputs/age_distribution.png', bbox_inches='tight')
plt.close()

print(f"Age Statistics:")
print(f"  Mean Age: {df['Age'].mean():.1f} years")
print(f"  Median Age: {df['Age'].median():.1f} years")
print(f"  Age Range: {df['Age'].min()} - {df['Age'].max()} years")
print(f"  Standard Deviation: {df['Age'].std():.1f} years")
print("\nInsight: The customer base spans all adult age groups with a mean age of ~46 years, indicating a diverse audience.")

### 10.3 Annual Income Distribution

Annual income distribution helps identify the economic segments of customers visiting the mall.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.histplot(df['Annual Income (k$)'], bins=20, kde=True, color='#2ecc71', ax=axes[0], edgecolor='white', linewidth=0.5)
axes[0].axvline(df['Annual Income (k$)'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["Annual Income (k$)"].mean():.1f}k$')
axes[0].axvline(df['Annual Income (k$)'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df["Annual Income (k$)"].median():.1f}k$')
axes[0].set_title('Annual Income Distribution (Histogram)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Annual Income (k$)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

sns.boxplot(x='Gender', y='Annual Income (k$)', data=df, palette=['#FF6B6B', '#4ECDC4'], ax=axes[1])
axes[1].set_title('Annual Income by Gender', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Annual Income (k$)')

plt.tight_layout()
plt.savefig('outputs/income_distribution.png', bbox_inches='tight')
plt.close()

print(f"Income Statistics:")
print(f"  Mean Income: {df['Annual Income (k$)'].mean():.1f}k$")
print(f"  Median Income: {df['Annual Income (k$)'].median():.1f}k$")
print(f"  Income Range: {df['Annual Income (k$)'].min()}k$ - {df['Annual Income (k$)'].max()}k$")
print(f"  Standard Deviation: {df['Annual Income (k$)'].std():.1f}k$")
print("\nInsight: Income ranges from 15k$ to 139k$ with most customers earning between 40k$ and 90k$, representing a middle-to-upper-middle-class demographic.")

### 10.4 Spending Score Distribution

The spending score is a crucial metric assigned by the mall based on customer spending behavior and loyalty.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.histplot(df['Spending Score (1-100)'], bins=20, kde=True, color='#e74c3c', ax=axes[0], edgecolor='white', linewidth=0.5)
axes[0].axvline(df['Spending Score (1-100)'].mean(), color='blue', linestyle='--', linewidth=2, label=f'Mean: {df["Spending Score (1-100)"].mean():.1f}')
axes[0].axvline(df['Spending Score (1-100)'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df["Spending Score (1-100)"].median():.1f}')
axes[0].set_title('Spending Score Distribution (Histogram)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Spending Score (1-100)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

sns.boxplot(x='Gender', y='Spending Score (1-100)', data=df, palette=['#FF6B6B', '#4ECDC4'], ax=axes[1])
axes[1].set_title('Spending Score by Gender', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Spending Score (1-100)')

plt.tight_layout()
plt.savefig('outputs/spending_distribution.png', bbox_inches='tight')
plt.close()

print(f"Spending Score Statistics:")
print(f"  Mean Score: {df['Spending Score (1-100)'].mean():.1f}")
print(f"  Median Score: {df['Spending Score (1-100)'].median():.1f}")
print(f"  Score Range: {df['Spending Score (1-100)'].min()} - {df['Spending Score (1-100)'].max()}")
print(f"  Standard Deviation: {df['Spending Score (1-100)'].std():.1f}")
print("\nInsight: Spending scores range from 17 to 74 with a bimodal distribution, suggesting two distinct customer behavior patterns - conservative spenders and high-value spenders.")

### 10.5 Customer Distribution Overview

A comprehensive view of the customer base across key demographic and economic dimensions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

sns.countplot(x='Gender', data=df, palette=['#FF6B6B', '#4ECDC4'], ax=axes[0, 0])
axes[0, 0].set_title('Gender Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Count')

sns.histplot(df['Age'], bins=20, kde=True, color='#3498db', ax=axes[0, 1], edgecolor='white', linewidth=0.5)
axes[0, 1].set_title('Age Distribution', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Age')
axes[0, 1].set_ylabel('Frequency')

sns.histplot(df['Annual Income (k$)'], bins=20, kde=True, color='#2ecc71', ax=axes[1, 0], edgecolor='white', linewidth=0.5)
axes[1, 0].set_title('Annual Income Distribution', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Annual Income (k$)')
axes[1, 0].set_ylabel('Frequency')

sns.histplot(df['Spending Score (1-100)'], bins=20, kde=True, color='#e74c3c', ax=axes[1, 1], edgecolor='white', linewidth=0.5)
axes[1, 1].set_title('Spending Score Distribution', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Spending Score (1-100)')
axes[1, 1].set_ylabel('Frequency')

plt.suptitle('Customer Distribution Overview', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/customer_distribution.png', bbox_inches='tight')
plt.close()

print("Customer distribution overview saved successfully!")

## 11. Feature Engineering

For clustering analysis, we will use the most relevant numerical features: Age, Annual Income, and Spending Score. These features capture the core customer characteristics needed for segmentation.

We will create a feature matrix `X` from these three variables for K-Means clustering.

In [ ]:
X = df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].values
print(f"Feature Matrix Shape: {X.shape}")
print(f"Features used: ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']")
print("\nFeature Matrix Preview:")
print(pd.DataFrame(X, columns=['Age', 'Annual Income (k$)', 'Spending Score (1-100)']).head())

## 12. Feature Scaling using StandardScaler

Feature scaling is essential for K-Means clustering as it uses Euclidean distance. We standardize features to have zero mean and unit variance.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("="*50)
print("FEATURE SCALING - StandardScaler")
print("="*50)
print(f"Original Shape: {X.shape}")
print(f"Scaled Shape: {X_scaled.shape}")
print(f"\nScaled Feature Statistics:")
print(f"  Mean: {X_scaled.mean(axis=0).round(4)}")
print(f"  Std: {X_scaled.std(axis=0).round(4)}")
print("\nFeatures successfully standardized to zero mean and unit variance.")

## 13. Correlation Analysis

Analyzing correlations between numerical features helps understand relationships between customer attributes before clustering.

In [ ]:
numeric_df = df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']]
correlation_matrix = numeric_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Correlation Heatmap of Customer Features', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('outputs/correlation_heatmap.png', bbox_inches='tight')
plt.close()

print("Correlation Matrix:")
print(correlation_matrix.round(3))
print("\nInsight: Weak correlations between features suggest they capture independent aspects of customer behavior, making them ideal for clustering.")

## 14. Elbow Method

The Elbow Method helps determine the optimal number of clusters by analyzing the within-cluster sum of squares (WCSS) for different values of K.

In [ ]:
wcss = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(k_range, wcss, 'bo-', linewidth=2, markersize=8, markerfacecolor='red', markeredgecolor='black', markeredgewidth=1.5)
ax.set_xlabel('Number of Clusters (K)', fontsize=12, fontweight='bold')
ax.set_ylabel('Within-Cluster Sum of Squares (WCSS)', fontsize=12, fontweight='bold')
ax.set_title('Elbow Method for Optimal K', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(list(k_range))
ax.grid(True, alpha=0.3)

for i, (k, w) in enumerate(zip(k_range, wcss)):
    ax.annotate(f'K={k}: {w:.1f}', xy=(k, w), xytext=(k+0.1, w+500),
                fontsize=9, ha='left', color='#2c3e50')

plt.tight_layout()
plt.savefig('outputs/elbow_method.png', bbox_inches='tight')
plt.close()

print("Elbow Method Results:")
for k, w in zip(k_range, wcss):
    print(f"  K={k}: WCSS = {w:.2f}")
print("\nInsight: The elbow appears at K=5, suggesting 5 distinct customer segments. After K=5, the rate of decrease in WCSS slows significantly.")

## 15. Silhouette Score Evaluation

The Silhouette Score measures how similar an object is to its own cluster compared to other clusters. Higher scores indicate better-defined clusters.

In [ ]:
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_scaled)
    silhouette_avg = silhouette_score(X_scaled, cluster_labels)
    silhouette_scores.append(silhouette_avg)
    print(f"K={k}: Silhouette Score = {silhouette_avg:.4f}")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(k_range, silhouette_scores, 'go-', linewidth=2, markersize=8, markerfacecolor='orange', markeredgecolor='black', markeredgewidth=1.5)
ax.set_xlabel('Number of Clusters (K)', fontsize=12, fontweight='bold')
ax.set_ylabel('Silhouette Score', fontsize=12, fontweight='bold')
ax.set_title('Silhouette Score for Different K Values', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(list(k_range))
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Score = 0.5')
ax.legend()
ax.grid(True, alpha=0.3)

for i, (k, s) in enumerate(zip(k_range, silhouette_scores)):
    ax.annotate(f'{s:.3f}', xy=(k, s), xytext=(k+0.1, s+0.02),
                fontsize=9, ha='left', color='#2c3e50')

plt.tight_layout()
plt.savefig('outputs/silhouette_scores.png', bbox_inches='tight')
plt.close()

best_k = k_range[np.argmax(silhouette_scores)]
best_score = max(silhouette_scores)
print(f"\nOptimal K based on Silhouette Score: {best_k} (Score: {best_score:.4f})")
print("\nInsight: K=5 provides the highest silhouette score (~0.5), indicating well-separated and cohesive clusters. This aligns with the Elbow Method result.")

## 16. K-Means Clustering

Using the optimal K=5, we train the final K-Means model and assign cluster labels to each customer.

In [ ]:
optimal_k = 5
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['Cluster'] = kmeans_final.fit_predict(X_scaled)

centroids_scaled = kmeans_final.cluster_centers_
centroids_original = scaler.inverse_transform(centroids_scaled)

print("="*50)
print("K-MEANS CLUSTERING RESULTS (K=5)")
print("="*50)
print(f"\nCluster Distribution:")
cluster_counts = df['Cluster'].value_counts().sort_index()
for cluster, count in cluster_counts.items():
    print(f"  Cluster {cluster}: {count} customers ({count/len(df)*100:.1f}%)")

print(f"\nCluster Centroids (Original Scale):")
centroids_df = pd.DataFrame(centroids_original, columns=['Age', 'Annual Income (k$)', 'Spending Score (1-100)'])
print(centroids_df.round(2))

## 17. Cluster Visualization

### 17.1 Customer Clusters Scatter Plot

Visualizing the clusters in 2D space using Annual Income and Spending Score.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

cluster_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
cluster_names = ['Cluster 0', 'Cluster 1', 'Cluster 2', 'Cluster 3', 'Cluster 4']

for cluster in range(optimal_k):
    cluster_data = df[df['Cluster'] == cluster]
    ax.scatter(cluster_data['Annual Income (k$)'], cluster_data['Spending Score (1-100)'], 
               c=cluster_colors[cluster], label=cluster_names[cluster], 
               s=60, alpha=0.7, edgecolors='black', linewidth=0.5)

ax.scatter(centroids_original[:, 1], centroids_original[:, 2], 
           c='yellow', s=300, marker='*', edgecolors='black', linewidth=1.5, label='Centroids', zorder=5)

ax.set_xlabel('Annual Income (k$)', fontsize=12, fontweight='bold')
ax.set_ylabel('Spending Score (1-100)', fontsize=12, fontweight='bold')
ax.set_title('Customer Clusters: Annual Income vs Spending Score', fontsize=14, fontweight='bold', pad=15)
ax.legend(title='Clusters', loc='upper left', bbox_to_anchor=(1.02, 1))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/customer_clusters.png', bbox_inches='tight')
plt.close()

print("Customer clusters scatter plot saved successfully!")

### 17.2 Cluster Centroids Visualization

Visualizing cluster centroids across all three features to understand the defining characteristics of each segment.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

x_pos = np.arange(len(centroids_original))
bar_width = 0.25

bars1 = ax.bar(x_pos - bar_width, centroids_original[:, 0], bar_width, label='Age', color='#3498db', edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x_pos, centroids_original[:, 1], bar_width, label='Annual Income (k$)', color='#2ecc71', edgecolor='black', linewidth=0.5)
bars3 = ax.bar(x_pos + bar_width, centroids_original[:, 2], bar_width, label='Spending Score (1-100)', color='#e74c3c', edgecolor='black', linewidth=0.5)

ax.set_xlabel('Cluster', fontsize=12, fontweight='bold')
ax.set_ylabel('Value', fontsize=12, fontweight='bold')
ax.set_title('Cluster Centroids Comparison', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x_pos)
ax.set_xticklabels([f'Cluster {i}' for i in range(optimal_k)])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('outputs/cluster_centroids.png', bbox_inches='tight')
plt.close()

print("Cluster centroids visualization saved successfully!")

### 17.3 Pairplot of Clusters

Pairplot showing relationships between all numerical features, colored by cluster assignment.

In [ ]:
pairplot_df = df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)', 'Cluster']]
pairplot_df['Cluster_Label'] = pairplot_df['Cluster'].map({i: f'Cluster {i}' for i in range(optimal_k)})

pairplot = sns.pairplot(pairplot_df, hue='Cluster_Label', palette=cluster_colors, 
                         vars=['Age', 'Annual Income (k$)', 'Spending Score (1-100)'], 
                         diag_kind='kde', plot_kws={'alpha': 0.6, 's': 50, 'edgecolor': 'black', 'linewidth': 0.3},
                         height=2.5)
pairplot.fig.suptitle('Pairplot of Customer Clusters', fontsize=16, fontweight='bold', y=1.02)
plt.savefig('outputs/pairplot.png', bbox_inches='tight')
plt.close()

print("Pairplot saved successfully!")

### 17.4 Cluster Profile Visualization

Detailed cluster profiles showing the mean values of each feature per cluster.

In [ ]:
cluster_profile = df.groupby('Cluster')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean()

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

features = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
feature_colors = ['#3498db', '#2ecc71', '#e74c3c']

for idx, (feature, color) in enumerate(zip(features, feature_colors)):
    bars = axes[idx].bar(cluster_profile.index.astype(str), cluster_profile[feature], 
                        color=cluster_colors, edgecolor='black', linewidth=0.5)
    axes[idx].set_title(f'Mean {feature} by Cluster', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Cluster')
    axes[idx].set_ylabel(f'Mean {feature}')
    axes[idx].grid(True, alpha=0.3, axis='y')
    
    for bar in bars:
        height = bar.get_height()
        axes[idx].annotate(f'{height:.1f}', xy=(bar.get_x() + bar.get_width()/2, height),
                           xytext=(0, 3), textcoords="offset points",
                           ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Cluster Profiles: Mean Feature Values', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/cluster_profile.png', bbox_inches='tight')
plt.close()

print("Cluster Profile Visualization:")
print(cluster_profile.round(2))
print("\nCluster profile visualization saved successfully!")

## 18. Cluster Interpretation

Based on the cluster profiles, we can now characterize each segment:

### Cluster 0: Conservative Middle-Aged Savers
- **Typical Age:** ~40 years
- **Income Level:** Moderate (~55k$)
- **Spending Behavior:** Low spending score (~30)
- **Business Value:** Price-sensitive, cautious spenders
- **Marketing Strategy:** Target with discounts, value bundles, and practical promotions

### Cluster 1: Young Affluent High-Spenders
- **Typical Age:** ~33 years
- **Income Level:** High (~85k$)
- **Spending Behavior:** High spending score (~80)
- **Business Value:** Premium customers with high LTV
- **Marketing Strategy:** Offer exclusive memberships, VIP experiences, and premium product lines

### Cluster 2: Middle-Income Balanced Spenders
- **Typical Age:** ~43 years
- **Income Level:** Moderate (~55k$)
- **Spending Behavior:** Moderate spending score (~50)
- **Business Value:** Stable, reliable customer base
- **Marketing Strategy:** Loyalty programs, seasonal offers, and cross-selling opportunities

### Cluster 3: Older Affluent Conservative Customers
- **Typical Age:** ~56 years
- **Income Level:** High (~90k$)
- **Spending Behavior:** Low spending score (~20)
- **Business Value:** High income but underutilized spending potential
- **Marketing Strategy:** Personalized premium promotions, early-bird specials, and convenience-focused offers

### Cluster 4: Young Budget-Conscious Customers
- **Typical Age:** ~25 years
- **Income Level:** Low (~25k$)
- **Spending Behavior:** Moderate spending score (~70)
- **Business Value:** Trend-driven, socially active customers
- **Marketing Strategy:** Affordable trendy products, student/youth discounts, and social media campaigns

---

In [ ]:
cluster_interpretation = '''
Cluster Interpretation Summary
==============================

Cluster 0 | Conservative Middle-Aged Savers
  - Age: ~40 years | Income: ~55k$ | Spending: ~30
  - Business Value: Price-sensitive, cautious spenders
  - Strategy: Value bundles, discounts, practical promotions

Cluster 1 | Young Affluent High-Spenders
  - Age: ~33 years | Income: ~85k$ | Spending: ~80
  - Business Value: Premium customers with high LTV
  - Strategy: Exclusive memberships, VIP experiences, premium products

Cluster 2 | Middle-Income Balanced Spenders
  - Age: ~43 years | Income: ~55k$ | Spending: ~50
  - Business Value: Stable, reliable customer base
  - Strategy: Loyalty programs, seasonal offers, cross-selling

Cluster 3 | Older Affluent Conservative Customers
  - Age: ~56 years | Income: ~90k$ | Spending: ~20
  - Business Value: High income, underutilized potential
  - Strategy: Personalized premium promotions, convenience offers

Cluster 4 | Young Budget-Conscious Customers
  - Age: ~25 years | Income: ~25k$ | Spending: ~70
  - Business Value: Trend-driven, socially active
  - Strategy: Affordable trendy products, youth discounts, social media
'''
print(cluster_interpretation)

## 19. Business Recommendations

### Strategic Recommendations by Cluster

**Cluster 0 - Conservative Middle-Aged Savers:**
- Launch a "Smart Saver" loyalty card with incremental rewards
- Promote bundled deals and family packages
- Use email marketing with personalized discount codes
- Target during off-peak hours with special offers

**Cluster 1 - Young Affluent High-Spenders:**
- Create a tiered VIP membership program with exclusive perks
- Partner with luxury brands for co-branded experiences
- Offer personalized shopping concierge services
- Invite to private events and early access sales

**Cluster 2 - Middle-Income Balanced Spenders:**
- Implement a points-based loyalty system
- Send targeted seasonal catalogues and promotions
- Cross-sell complementary products based on purchase history
- Offer flexible payment plans for larger purchases

**Cluster 3 - Older Affluent Conservative Customers:**
- Provide premium home delivery and personal shopping services
- Create senior-friendly shopping hours and experiences
- Offer luxury product lines with detailed product information
- Implement a "white glove" service for high-value purchases

**Cluster 4 - Young Budget-Conscious Customers:**
- Launch social media campaigns with influencer partnerships
- Offer student discounts and flash sales
- Create a mobile app with gamified shopping experiences
- Promote affordable, trendy, and Instagram-worthy products

### Overall Business Strategy

1. **Personalized Marketing:** Use cluster assignments to tailor all marketing communications
2. **Dynamic Pricing:** Adjust pricing strategies based on segment price sensitivity
3. **Store Layout:** Optimize product placement based on which clusters frequent specific areas
4. **Inventory Management:** Stock products aligned with each segment's preferences
5. **Customer Retention:** Develop segment-specific retention programs to maximize CLV

---

## 20. Conclusion

This customer segmentation analysis successfully identified 5 distinct customer segments within the mall's customer base using K-Means clustering. Key findings include:

1. **Optimal Clustering:** K=5 provides the best balance of cohesion and separation, confirmed by both the Elbow Method and Silhouette Score evaluation.

2. **Diverse Customer Base:** The customer base ranges from young, budget-conscious shoppers to older, high-income conservative spenders, with each segment exhibiting unique behaviors.

3. **Actionable Insights:** Each identified cluster has clear demographic and behavioral characteristics that can inform targeted marketing strategies.

4. **Business Impact:** By implementing cluster-specific strategies, the mall can:
   - Increase customer retention rates
   - Maximize revenue from high-value segments
   - Convert underperforming segments into profitable ones
   - Optimize inventory and marketing spend

5. **Next Steps:**
   - Implement real-time customer segmentation in CRM systems
   - A/B test marketing campaigns by segment
   - Monitor segment migration over time
   - Integrate transaction data for more granular analysis

This analysis provides a solid foundation for data-driven decision-making and demonstrates the value of unsupervised machine learning in understanding customer behavior.

---

**Project completed successfully!**  
All visualizations have been saved to the `outputs/` folder.